# ESE 670 (12pt) — Homework 4: optimal transport and bridge matching

**Part B (5 pts, GPU).** Image inpainting on CIFAR-10 ($32\times32$, colour) with bridge matching. We train four small models
that differ only in **where the randomness comes from**: none, drawn once at the start, injected continuously (a Brownian bridge),
or an unconditional generator from pure noise. We compare their samples, their diversity, their errors and their behaviour with
few steps.

All code is given and runs as is. The questions are in the markdown cells; answer them in your own cells (derivations, a few sentences). Part B trains four networks of about 6M parameters each: roughly 10–15 minutes per network on an A100, several times longer on a T4. Checkpoints are saved in `ckpt/`, so a restarted runtime does not retrain.

# Part B. Inpainting with bridge matching

**Data.** CIFAR-10 images $x_1\in[-1,1]^{3\times32\times32}$ and a mask $m\in\{0,1\}^{32\times32}$ ($1$ = missing). Training masks
are random rectangles with sides 8–16 pixels; for evaluation we use a centred $14\times14$ square.

**Four models, one recipe.** A start $x_0$ is built from the masked image, and the path between $x_0$ and $x_1$ is a Brownian
bridge with noise level $\sigma$:
$$x_t=(1-t)\,x_0+t\,x_1+\sigma\sqrt{t(1-t)}\,z,\qquad z\sim\mathcal N(0,I).$$
A network $\hat x_{1,\theta}(x_t,t,\cdot)$ predicts $x_1$ (loss $\|\hat x_{1,\theta}-x_1\|^2$), and we sample with the SDE
$\mathrm dX_t=\frac{\hat x_{1,\theta}(X_t,t)-X_t}{1-t}\,\mathrm dt+\sigma\,\mathrm dW_t$ from $X_0=x_0$ (Euler–Maruyama).

| model | start $x_0$ | $\sigma$ | network input | where the randomness comes from |
|---|---|---|---|---|
| `ode_zero` | masked image, hole $=0$ | $0$ | $x_t,\ x_0,\ m$ | nowhere |
| `ode_noise` | masked image, hole $=$ noise | $0$ | $x_t,\ x_0,\ m$ | drawn once, at $t=0$ (flow matching) |
| `bridge` | masked image, hole $=0$ | $1$ | $x_t,\ x_0,\ m$ | injected along the path (Brownian bridge) |
| `uncond` | pure noise | $0$ | $x_t$ only | at $t=0$; inpainting by replacing the known pixels |

The three conditional models are **augmented** bridge matching (the network sees $x_0$), and outside the hole their prediction is
set to the known pixels: $\hat x_1=m\odot\text{net}+(1-m)\odot x_0$. The loss is then taken on the hole only. In other words, we run diffusion process from $m$, where $m$ is either one point (matrix of zerows), or a sample from standard Gaussian.

In [7]:
import math, copy, os, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as TF

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")          # TF32 on A100
amp_dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

torch.manual_seed(0)

norm = TF.Compose([TF.ToTensor(), TF.Normalize((0.5,) * 3, (0.5,) * 3)])
train_set = torchvision.datasets.CIFAR10("data", train=True, download=True,
                                         transform=TF.Compose([TF.RandomHorizontalFlip(), norm]))
test_set = torchvision.datasets.CIFAR10("data", train=False, download=True, transform=norm)
#loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
loader = torch.utils.data.DataLoader(
    train_set, batch_size=128, shuffle=True, drop_last=True,
    num_workers=min(8, os.cpu_count()), pin_memory=True, persistent_workers=True)

def to_img(x):                       # [-1, 1] tensor (3, H, W) -> numpy image in [0, 1]
    return ((x.clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()

# def random_box_masks(B, lo=8, hi=16, H=32):
#     m = torch.zeros(B, 1, H, H)
#     for i in range(B):
#         h, w = torch.randint(lo, hi + 1, (2,)).tolist()
#         y, x = torch.randint(0, H - h + 1, (1,)).item(), torch.randint(0, H - w + 1, (1,)).item()
#         m[i, 0, y:y + h, x:x + w] = 1
#     return m
def random_box_masks(B, lo=8, hi=16, H=32, device="cpu"):
    h = torch.randint(lo, hi + 1, (B,), device=device)
    w = torch.randint(lo, hi + 1, (B,), device=device)
    y = (torch.rand(B, device=device) * (H - h + 1)).long()
    x = (torch.rand(B, device=device) * (H - w + 1)).long()
    r = torch.arange(H, device=device)
    rows = (r >= y[:, None]) & (r < (y + h)[:, None])      # (B, H)
    cols = (r >= x[:, None]) & (r < (x + w)[:, None])      # (B, W)
    return (rows[:, :, None] & cols[:, None, :]).float()[:, None]

def center_masks(B, size=14, H=32):
    m = torch.zeros(B, 1, H, H)
    a = (H - size) // 2
    m[:, :, a:a + size, a:a + size] = 1
    return m

def make_start(x1, m, fill):
    if fill == "zero":  return x1 * (1 - m)
    if fill == "noise": return x1 * (1 - m) + m * torch.randn_like(x1)
    if fill == "pure":  return torch.randn_like(x1)
    raise ValueError(fill)

CONFIGS = {
    "ode_zero":  dict(fill="zero",  sigma=0.0, cond=True),
    "ode_noise": dict(fill="noise", sigma=0.0, cond=True),
    "bridge":    dict(fill="zero",  sigma=1.0, cond=True),
    "uncond":    dict(fill="pure",  sigma=0.0, cond=False),
}

xb, _ = next(iter(loader)); mb = random_box_masks(8)
fig, ax = plt.subplots(3, 8, figsize=(13, 5))
for j in range(8):
    for i, fill in enumerate(["zero", "noise", "pure"]):
        ax[i, j].imshow(to_img(make_start(xb[j:j + 1], mb[j:j + 1], fill)[0])); ax[i, j].axis("off")
for i, fill in enumerate(["start: hole = 0", "start: hole = noise", "start: pure noise"]):
    ax[i, 0].set_title(fill, fontsize=9, loc="left")
plt.tight_layout(); plt.show()

### The network
A small U-Net (6M parameters) with a sinusoidal embedding of $t$. Conditional models get $7$ input channels
($x_t$, $x_0$, $m$), the unconditional one gets $3$.

In [8]:
class SinEmb(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, t):                                  # t: (B,) in [0, 1]
        half = self.dim // 2
        f = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device) / half)
        a = 1000.0 * t[:, None] * f[None]
        return torch.cat([torch.sin(a), torch.cos(a)], dim=1)

class ResBlock(nn.Module):
    def __init__(self, cin, cout, tdim, dropout=0.1):
        super().__init__()
        self.n1, self.c1 = nn.GroupNorm(32, cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.t = nn.Linear(tdim, cout)
        self.n2, self.c2 = nn.GroupNorm(32, cout), nn.Conv2d(cout, cout, 3, padding=1)
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, temb):
        h = self.c1(F.silu(self.n1(x)))
        h = h + self.t(F.silu(temb))[:, :, None, None]
        h = self.c2(self.drop(F.silu(self.n2(h))))
        return h + self.skip(x)

class Down(nn.Module):
    def __init__(self, c):
        super().__init__(); self.conv = nn.Conv2d(c, c, 3, stride=2, padding=1)
    def forward(self, x): return self.conv(x)

class Up(nn.Module):
    def __init__(self, c):
        super().__init__(); self.conv = nn.Conv2d(c, c, 3, padding=1)
    def forward(self, x): return self.conv(F.interpolate(x, scale_factor=2, mode="nearest"))

class UNet(nn.Module):
    def __init__(self, in_ch, out_ch=3, ch=64, mults=(1, 2, 2), nres=2, tdim=256):
        super().__init__()
        self.temb = nn.Sequential(SinEmb(ch), nn.Linear(ch, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.inc = nn.Conv2d(in_ch, ch, 3, padding=1)
        self.down, chans, c = nn.ModuleList(), [ch], ch
        for i, mlt in enumerate(mults):
            for _ in range(nres):
                self.down.append(ResBlock(c, ch * mlt, tdim)); c = ch * mlt; chans.append(c)
            if i < len(mults) - 1:
                self.down.append(Down(c)); chans.append(c)
        self.mid = nn.ModuleList([ResBlock(c, c, tdim), ResBlock(c, c, tdim)])
        self.up = nn.ModuleList()
        for i, mlt in reversed(list(enumerate(mults))):
            for _ in range(nres + 1):
                self.up.append(ResBlock(c + chans.pop(), ch * mlt, tdim)); c = ch * mlt
            if i > 0:
                self.up.append(Up(c))
        self.out = nn.Sequential(nn.GroupNorm(32, c), nn.SiLU(), nn.Conv2d(c, out_ch, 3, padding=1))

    def forward(self, x, t):
        temb = self.temb(t)
        h = self.inc(x); hs = [h]
        for layer in self.down:
            h = layer(h, temb) if isinstance(layer, ResBlock) else layer(h)
            hs.append(h)
        for layer in self.mid:
            h = layer(h, temb)
        for layer in self.up:
            h = layer(torch.cat([h, hs.pop()], 1), temb) if isinstance(layer, ResBlock) else layer(h)
        return self.out(h)

print("parameters:", sum(p.numel() for p in UNet(7).parameters()) / 1e6, "M")

### Training
`predict_x1` implements $\hat x_1$ for both kinds of models. `bridge_sample` draws $x_t$ from the Brownian bridge between $x_0$ (in practice, $m$) and
$x_1$. The loss is the squared error on the hole for the conditional models and on the whole image for `uncond`.

In [9]:
def predict_x1(model, cfg, x, t, x0, m):
    tt = torch.full((x.shape[0],), float(t), device=x.device) if not torch.is_tensor(t) else t
    if cfg["cond"]:
        out = model(torch.cat([x, x0, m], 1), tt)
        return m * out + (1 - m) * x0                 # the known pixels are known
    return model(x, tt)

def bridge_sample(x0, x1, t, sigma):
    t4 = t[:, None, None, None]
    return (1 - t4) * x0 + t4 * x1 + sigma * torch.sqrt(t4 * (1 - t4)) * torch.randn_like(x1)

def loss_fn(model, cfg, x1):
    B = x1.shape[0]
    #m = random_box_masks(B).to(device)
    m = random_box_masks(B, device=device)
    x0 = make_start(x1, m, cfg["fill"])
    t = torch.rand(B, device=device)
    xt = bridge_sample(x0, x1, t, cfg["sigma"])
    err = (predict_x1(model, cfg, xt, t, x0, m) - x1) ** 2
    if cfg["cond"]:
        return (err * m).sum() / (3 * m.sum())
    return err.mean()

def infinite(loader):
    while True:
        for x, _ in loader:
            yield x


def train(name, steps=20000, lr=2e-4, ema_decay=0.9995, save_every=2000):
    path, last = f"ckpt/{name}.pt", f"ckpt/{name}_last.pt"
    cfg = CONFIGS[name]
    model = UNet(7 if cfg["cond"] else 3).to(device)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device)); model.eval()
        print(f"{name}: loaded {path}"); return model
    ema = copy.deepcopy(model).eval()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)
    start = 1
    if os.path.exists(last):                                   # resume an interrupted run
        ck = torch.load(last, map_location=device)
        model.load_state_dict(ck["model"]); ema.load_state_dict(ck["ema"])
        opt.load_state_dict(ck["opt"]); start = ck["it"] + 1
        print(f"{name}: resuming from step {ck['it']}")
    os.makedirs("ckpt", exist_ok=True)
    mp, ep = list(model.parameters()), list(ema.parameters())
    data_iter, t0 = infinite(loader), time.time()
    run = torch.zeros((), device=device)
    for it in range(start, steps + 1):
        x1 = next(data_iter).to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=use_amp):
            loss = loss_fn(model, cfg, x1)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(mp, 1.0)
        scaler.step(opt); scaler.update()
        with torch.no_grad():
            torch._foreach_mul_(ep, ema_decay)
            torch._foreach_add_(ep, mp, alpha=1 - ema_decay)
            for be, b in zip(ema.buffers(), model.buffers()):
                be.copy_(b)
        run = 0.99 * run + 0.01 * loss.detach()
        if it % save_every == 0:
            print(f"{name}  step {it:6d}  loss {run.item() / (1 - 0.99 ** (it - start + 1)):.4f}  {time.time() - t0:5.0f}s")
            torch.save({"model": model.state_dict(), "ema": ema.state_dict(),
                        "opt": opt.state_dict(), "it": it}, last)
    torch.save(ema.state_dict(), path)
    if os.path.exists(last): os.remove(last)
    return ema

models = {}
for name in CONFIGS:
    models[name] = train(name, steps=30000 if name == "uncond" else 20000)

### Sampling
Euler–Maruyama on a uniform grid $0=t_0<\dots<t_n=1$:
$X_{k+1}=X_k+\frac{\hat x_1(X_k,t_k)-X_k}{1-t_k}\,\Delta t+\sigma\sqrt{\Delta t}\,\xi_k$, with no noise in the last step.
For `uncond` the known pixels are overwritten after every step by their own straight-line path
$(1-t_{k+1})\,x_0+t_{k+1}\,y$, where $y$ is the observed image (the replacement trick of RePaint).

In [10]:
@torch.no_grad()
def sample(name, y, m, n_steps=100, return_path=False):
    # y: observed images (the hole content is ignored), m: masks; returns samples (and the path if asked)
    cfg, model = CONFIGS[name], models[name]
    x0 = make_start(y, m, cfg["fill"])
    x, ts, path = x0.clone(), torch.linspace(0, 1, n_steps + 1).tolist(), [x0.clone()]
    for k in range(n_steps):
        t, dt = ts[k], ts[k + 1] - ts[k]
        x1_hat = predict_x1(model, cfg, x, t, x0, m)
        x = x + (x1_hat - x) / (1 - t) * dt               # the last step lands on x1_hat exactly
        if cfg["sigma"] > 0 and k < n_steps - 1:
            x = x + cfg["sigma"] * math.sqrt(dt) * torch.randn_like(x)
        if not cfg["cond"]:                                # replacement for the unconditional model
            known = (1 - ts[k + 1]) * x0 + ts[k + 1] * y
            x = m * x + (1 - m) * known
        if return_path:
            path.append(x.clone())
    return (x, torch.stack(path)) if return_path else x

torch.manual_seed(1)
idx = [3, 10, 21, 44, 57, 90]
y = torch.stack([test_set[i][0] for i in idx]).to(device)
m = center_masks(len(idx)).to(device)
S = 4                                                    # samples per image
rows = [("ground truth", y), ("observed", y * (1 - m))]
for name in CONFIGS:
    for s in range(S if name != "ode_zero" else 1):
        rows.append((f"{name} #{s + 1}", sample(name, y, m)))

fig, ax = plt.subplots(len(rows), len(idx), figsize=(1.6 * len(idx), 1.6 * len(rows)))
for r, (title, imgs) in enumerate(rows):
    for j in range(len(idx)):
        ax[r, j].imshow(to_img(imgs[j])); ax[r, j].set_xticks([]); ax[r, j].set_yticks([])
    ax[r, 0].set_ylabel(title, fontsize=8, rotation=0, ha="right", va="center")
plt.tight_layout(); plt.show()

### Paths of one pixel
The red channel of the central pixel of the hole, for 16 samples of the same image, against $t$. Compare with the bridge figure of
the lecture. Note, that the last Euler step lands exactly on $\hat x_1(X_{n-1},t_{n-1})$. At $k=n-1$, $\Delta t=1-t_{n-1}$, so $X_n=X_{n-1}+(\hat x_1-X_{n-1})=\hat x_1$. The bridge has
$\operatorname{Var}(X_t\mid x_0,x_1)=\sigma^2t(1-t)\to0$ at $t=1$: the endpoint is pinned, so the last step must not add
$\sigma\sqrt{\Delta t}$ of noise (it would be left in the output).

In [ ]:
torch.manual_seed(2)
y1, mk1 = y[:1].repeat(16, 1, 1, 1), m[:1].repeat(16, 1, 1, 1)
fig, ax = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
tgrid = np.linspace(0, 1, 101)
for a, name in zip(ax, CONFIGS):
    _, P = sample(name, y1, mk1, n_steps=100, return_path=True)
    a.plot(tgrid, P[:, :, 0, 16, 16].cpu().numpy(), lw=0.8)
    a.axhline(y1[0, 0, 16, 16].item(), color="k", ls="--", lw=1, label="ground truth")
    a.set(title=name, xlabel="t"); a.legend(fontsize=8)
ax[0].set_ylabel("red channel, centre pixel")
plt.tight_layout(); plt.show()

**Question 6 (1 pts).**
In the pixel paths, which models give one path, which give a fan, and when does the fan open (at $t=0$ or gradually)?

_Answer_

### Errors and diversity
For 256 test images with the centred mask: the PSNR of the hole against the ground truth, averaged over 4 samples per image, and the
**diversity**, the standard deviation over the 4 samples averaged over the hole. The last column is the PSNR of the **average** of
the 4 samples.

In [ ]:
def hole_mse(a, b, m):                  # per-image MSE over the hole, on the [0, 1] scale
    return (((a - b) / 2) ** 2 * m).sum((1, 2, 3)) / (3 * m.sum((1, 2, 3)))

def psnr_hole(a, b, m):                 # images in [-1, 1]; PSNR of the hole on the [0, 1] scale
    return (10 * torch.log10(1 / hole_mse(a, b, m))).mean().item()

def db(mse):                            # MSE on the [0, 1] scale -> dB
    return 10 * math.log10(1 / mse)

@torch.no_grad()
def evaluate(name, n_img=256, S=4, n_steps=100):
    torch.manual_seed(3)
    yy = torch.stack([test_set[i][0] for i in range(n_img)]).to(device)
    mm = center_masks(n_img).to(device)
    xs = torch.stack([sample(name, yy, mm, n_steps) for _ in range(S)])      # (S, B, 3, H, W)
    xbar = xs.mean(0)
    r = dict(
        psnr=np.mean([psnr_hole(x, yy, mm) for x in xs]),
        psnr_mean=psnr_hole(xbar, yy, mm),
        div=((xs.std(0) * mm).sum() / (3 * mm.sum())).item() if S > 1 else 0.0,
        mse1=np.mean([hole_mse(x, yy, mm).mean().item() for x in xs]),       # one sample
        mseS=hole_mse(xbar, yy, mm).mean().item(),                           # mean of S samples
    )
    # exact identity: mse1 = mseS + (S-1)/S * V, with V the (unbiased) sample variance over the hole
    r["V"] = S / (S - 1) * (r["mse1"] - r["mseS"]) if S > 1 else 0.0
    r["floor"] = r["mse1"] - r["V"]        # = MMSE + bias^2: the error of the model's mean with S -> infinity
    r["ratio"] = r["V"] / r["floor"]       # perfect sampler: bias = 0 and V = MMSE, so ratio = 1
    return r

S = 4
print(f"{'model':10s} {'PSNR':>6s} {'div':>6s} {'PSNR':>7s} | {'MSE':>7s} {'MSE':>7s} {'V':>7s} {'MMSE+':>7s} {'PSNR':>7s} {'V /':>6s}")
print(f"{'':10s} {'1 smp':>6s} {'':>6s} {f'mean {S}':>7s} | {'1 smp':>7s} {f'mean {S}':>7s} {'':>7s} {'bias^2':>7s} {'S->inf':>7s} {'floor':>6s}")
results = {}
for name in CONFIGS:
    r = results[name] = evaluate(name, S=S)
    print(f"{name:10s} {r['psnr']:6.2f} {r['div']:6.3f} {r['psnr_mean']:7.2f} | "
          f"{r['mse1']:7.4f} {r['mseS']:7.4f} {r['V']:7.4f} {r['floor']:7.4f} {db(r['floor']):7.2f} {r['ratio']:6.2f}")

# A perfect posterior sampler has bias = 0 and V = MMSE, so for it
#   PSNR(1 sample) = PSNR(posterior mean) - 3.01 dB,   PSNR(mean of S) = PSNR(posterior mean) - 10 log10(1 + 1/S)
# and V / floor = 1.  V / floor < 1: too little variety or a systematic error (bias); > 1: more variety than the context allows.
# The deterministic ode_zero has V = 0, so its floor is its own MSE: compare it with the floors of the samplers.

The model `ode_zero` has $\sigma=0$ and a deterministic start $x_0$. For $t>0$ the training pair satisfies $x_1=x_0+(x_t-x_0)/t$, so the optimal predictor is $\hat x_1^\ast(x,t)=\mathbb E[x_1\mid x_t,x_0]=x_0+(x-x_0)/t$ on the support, while $\hat x_1^\ast(x_0,0)=\mathbb E[x_1\mid x_0]$.
The ODE $\dot x=(\hat x_1^\ast(x,t)-x)/(1-t)$ started at $x_0$ moves on the straight line
$x_t=x_0+t\,(\mathbb E[x_1\mid x_0]-x_0)$ and outputs the **posterior mean** $\mathbb E[x_1\mid x_0]$. The average of samples from $p(x_1\mid x_0)$ estimates $\mathbb E[x_1\mid x_0]$.

**Question 7 (2 pts).**  Use the table PSNR (peak signal-to-noise ratio is the mean squared error on a log scale, measured relative to the largest possible pixel value:
$PSNR = 10 \log_{10}\frac{MAX^2}{MSE},$ $MSE = \frac{1}{n}\sum_i (\hat{x}_i - x_i)^2$).Is the PSNR of `ode_zero` close to the PSNR of the average of the `bridge` samples? Why should it be?

_Answer_

### Few steps
The same images with $n\in\{1,2,5,20,100\}$ sampler steps.

In [ ]:
torch.manual_seed(4)
steps_list = [1, 2, 5, 20, 100]
names = ["ode_noise", "bridge", "uncond"]
fig, ax = plt.subplots(len(names) * 3, len(steps_list), figsize=(1.7 * len(steps_list), 1.7 * 3 * len(names)))
for r, name in enumerate(names):
    for j, n in enumerate(steps_list):
        torch.manual_seed(5)
        out = sample(name, y[:3], m[:3], n_steps=n)
        for k in range(3):
            a = ax[3 * r + k, j]; a.imshow(to_img(out[k])); a.set_xticks([]); a.set_yticks([])
            if k == 0: a.set_title(f"{name}, n = {n}", fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'model':10s}" + "".join(f"  n={n:<4d}" for n in steps_list) + "   (PSNR of the hole, 128 images, 1 sample)")
for name in names:
    row = [evaluate(name, n_img=128, S=1, n_steps=n)["psnr"] for n in steps_list]
    print(f"{name:10s}" + "".join(f"  {v:6.2f}" for v in row))

**Question 9 (2 pts).**
(a) What does every model output with $n=1$? (Look at the first Euler step and at Question 7.) Is this a sample?

(b) Which model degrades more gracefully as $n$ decreases, `ode_noise` or `bridge`? (Think about the shape of the paths and the size of the injected noise $\sigma\sqrt{\Delta t}$ compared with what the drift can remove in one step).

_Answer_